# Binary classification of `PW_0.5h`

Predict whether a failure occurs within the next 30 minutes from lagged sensor readings.

1. Read `PW_0.5h/train.csv` and `PW_0.5h/val.csv`.
2. Expand each series with `lag_window`. Each row keeps the last `READING_WINDOW` minutes of the sensors. `PW_0.5h` stays the label of the newest minute and is not used as a feature.
3. Hold out the last 20% of the lagged training rows as an eval set.
4. Tune the tree hyperparameters with Optuna's Bayesian sampler and expanding-window time-series cross-validation at learning rate 0.8. Fit that model once at a low learning rate and take the tree count from early stopping on the eval holdout. Refit the final model on the fit rows with the stage-1 parameters, that learning rate, and that tree count.
5. Choose the F1 threshold on the eval holdout, then report F1 and AUC on `train.csv` and `val.csv`.

In [21]:
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

from pipeline_functions import lag_window

DATA_DIR = Path("PW_0.5h")
TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TARGET = "PW_0.5h"

READING_WINDOW = 30
EVAL_FRACTION = 0.20
N_SPLITS = 5
N_TRIALS = 40
RANDOM_SEED = 42

STAGE1_LEARNING_RATE = 0.8
STAGE1_N_ESTIMATORS = 500
STAGE2_LEARNING_RATE = 0.05
STAGE2_N_ESTIMATORS = 2000
EARLY_STOPPING_ROUNDS = 40

## Load and lag

`READING_WINDOW` is 30 minutes, the same length as the failure horizon. Each file is lagged on its own, in timestamp order.

In [22]:
def load_series(path):
    frame = pd.read_csv(path, parse_dates=["timestamp"])
    return frame.set_index("timestamp").sort_index()


def summarize(name, frame):
    labels = frame[TARGET].astype(int)
    print(
        f"{name}: {len(frame):,} rows, {frame.shape[1] - 1} features, "
        f"{int(labels.sum()):,} positives ({labels.mean():.2%}), "
        f"{frame.index.min()} to {frame.index.max()}"
    )


train_raw = load_series(TRAIN_PATH)
val_raw = load_series(VAL_PATH)
train = lag_window(train_raw, READING_WINDOW)
val = lag_window(val_raw, READING_WINDOW)

summarize("train.csv", train)
summarize("val.csv", val)
lag0_columns = [column for column in train.columns if column.endswith("_lag_0") or column == TARGET]
train.loc[:, lag0_columns].head(3)

train.csv: 40,325 rows, 120 features, 810 positives (2.01%), 2023-08-01 03:34:00 to 2023-08-29 03:38:00
val.csv: 34,315 rows, 120 features, 540 positives (1.57%), 2023-08-29 04:08:00 to 2023-09-22 00:02:00


,CMS air pressure_lag_0,Concentrazione ossigeno base_lag_0,Nitrogen pressure_lag_0,Oxygen over threshold_lag_0,PW_0.5h
timestamp,,,,,
2023-08-01 03:34:00,-0.578901,-0.986817,-1.394478,-0.044569,0
2023-08-01 03:35:00,-0.578901,-0.986817,-1.394478,-0.044569,0
2023-08-01 03:36:00,1.270593,-0.986817,-1.394478,-0.044569,0


## Eval holdout

The eval set is the last 20% of lagged `train.csv`. It is used to pick the learning rate, the number of trees, and the F1 threshold. Time-series cross-validation uses only the earlier fit rows. `val.csv` stays out of fitting and tuning.

In [23]:
def features_and_labels(frame):
    labels = frame[TARGET].astype(int)
    features = frame.drop(columns=[TARGET])
    return features, labels


X_train, y_train = features_and_labels(train)
eval_start = int(len(train) * (1 - EVAL_FRACTION))
if eval_start <= 0 or eval_start >= len(train):
    raise ValueError("Training series is too short for the eval holdout")

X_fit, y_fit = X_train.iloc[:eval_start], y_train.iloc[:eval_start]
X_eval, y_eval = X_train.iloc[eval_start:], y_train.iloc[eval_start:]
feature_columns = list(X_fit.columns)

SCALE_POS_WEIGHT = float((y_fit == 0).sum() / (y_fit == 1).sum())
print(
    f"fit rows {len(X_fit):,} ({int(y_fit.sum())} positives), "
    f"eval rows {len(X_eval):,} ({int(y_eval.sum())} positives)"
)
print(f"eval starts at {X_eval.index[0]}")
print(f"scale_pos_weight {SCALE_POS_WEIGHT:.2f}")

fit rows 32,260 (600 positives), eval rows 8,065 (210 positives)
eval starts at 2023-08-23 13:14:00
scale_pos_weight 52.77


## Stage 1: tree hyperparameters at a high learning rate

The learning rate stays at 0.8. Optuna's TPE sampler searches depth, child weight, subsample, column sample, gamma, and L2 jointly. Each trial is scored by expanding-window time-series cross-validation from `TimeSeriesSplit`. Every candidate stops early on F1. The score is the mean validation F1, using the threshold on that fold's validation rows that maximizes F1.

In [24]:
def make_classifier(
    tree_params,
    learning_rate,
    n_estimators,
    early_stopping_rounds=None,
    eval_metric="auc",
):
    settings = dict(
        objective="binary:logistic",
        eval_metric=eval_metric,
        tree_method="hist",
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        scale_pos_weight=SCALE_POS_WEIGHT,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0,
        **tree_params,
    )
    if early_stopping_rounds is not None:
        settings["early_stopping_rounds"] = early_stopping_rounds
    return XGBClassifier(**settings)


def fold_indices(n_rows):
    splitter = TimeSeriesSplit(n_splits=N_SPLITS)
    for train_idx, valid_idx in splitter.split(np.arange(n_rows)):
        if y_fit.iloc[valid_idx].nunique() < 2:
            raise ValueError("A time-series fold has one validation class")
        yield train_idx, valid_idx


def best_f1(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)
    grid = np.linspace(0.01, 0.99, 99)
    predicted = proba[None, :] >= grid[:, None]
    positive = y_true == 1
    true_positive = (predicted & positive).sum(axis=1)
    false_positive = (predicted & ~positive).sum(axis=1)
    false_negative = (~predicted & positive).sum(axis=1)
    precision = np.divide(
        true_positive,
        true_positive + false_positive,
        out=np.zeros(len(grid)),
        where=(true_positive + false_positive) > 0,
    )
    recall = np.divide(
        true_positive,
        true_positive + false_negative,
        out=np.zeros(len(grid)),
        where=(true_positive + false_negative) > 0,
    )
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros(len(grid)), where=denom > 0)
    return float(f1.max())


def neg_best_f1(y_true, y_score):
    # XGBoost minimizes a custom metric, so early stopping sees negative F1.
    return -best_f1(y_true, y_score)


def mean_cv_f1(tree_params):
    scores = []
    trees = []
    for train_idx, valid_idx in fold_indices(len(X_fit)):
        model = make_classifier(
            tree_params,
            learning_rate=STAGE1_LEARNING_RATE,
            n_estimators=STAGE1_N_ESTIMATORS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            eval_metric=neg_best_f1,
        )
        model.fit(
            X_fit.iloc[train_idx],
            y_fit.iloc[train_idx],
            eval_set=[(X_fit.iloc[valid_idx], y_fit.iloc[valid_idx])],
            verbose=False,
        )
        proba = model.predict_proba(X_fit.iloc[valid_idx])[:, 1]
        scores.append(best_f1(y_fit.iloc[valid_idx], proba))
        trees.append(model.best_iteration + 1)
    return float(np.mean(scores)), float(np.mean(trees))


def objective(trial):
    candidate = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 20.0, log=True),
    }
    score, trees = mean_cv_f1(candidate)
    trial.set_user_attr("mean_best_trees", trees)
    print(f"trial {trial.number:02d}  cv_f1={score:.4f}  trees={trees:.0f}  {candidate}")
    return score


print("Time-series folds:")
for fold, (train_idx, valid_idx) in enumerate(fold_indices(len(X_fit)), start=1):
    print(
        f"  fold {fold}: train {len(train_idx):,} ({int(y_fit.iloc[train_idx].sum())} positives), "
        f"valid {len(valid_idx):,} ({int(y_fit.iloc[valid_idx].sum())} positives)"
    )

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED, multivariate=True),
)
started = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS)
tree_params = study.best_params
print(f"\nstage 1 finished in {time.perf_counter() - started:.0f}s")
print(f"best cv_f1={study.best_value:.4f}")
print("selected tree parameters:")
print(tree_params)

trial_results = (
    pd.DataFrame(
        [
            {
                **trial.params,
                "cv_f1": trial.value,
                "mean_best_trees": trial.user_attrs["mean_best_trees"],
            }
            for trial in study.trials
            if trial.value is not None
        ]
    )
    .sort_values("cv_f1", ascending=False)
    .reset_index(drop=True)
)
trial_results

Time-series folds:
  fold 1: train 5,380 (120 positives), valid 5,376 (60 positives)
  fold 2: train 10,756 (180 positives), valid 5,376 (120 positives)
  fold 3: train 16,132 (300 positives), valid 5,376 (60 positives)
  fold 4: train 21,508 (360 positives), valid 5,376 (120 positives)
  fold 5: train 26,884 (480 positives), valid 5,376 (120 positives)


/Users/kingsleychan/Projects/IBM Z Datathon/IBM_Z_Datathon/kingsleyenv/lib/python3.13/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(


trial 00  cv_f1=0.5130  trees=6  {'max_depth': 5, 'min_child_weight': 20, 'subsample': 0.9195981825434215, 'colsample_bytree': 0.8795975452591109, 'gamma': 0.15601864044243652, 'reg_lambda': 1.5957084694148351}
trial 01  cv_f1=0.5388  trees=27  {'max_depth': 3, 'min_child_weight': 18, 'subsample': 0.8803345035229626, 'colsample_bytree': 0.9124217733388136, 'gamma': 0.020584494295802447, 'reg_lambda': 18.276027831785726}
trial 02  cv_f1=0.5240  trees=6  {'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.7545474901621302, 'colsample_bytree': 0.7550213529560301, 'gamma': 0.3042422429595377, 'reg_lambda': 4.816414530907083}
trial 03  cv_f1=0.5406  trees=15  {'max_depth': 5, 'min_child_weight': 6, 'subsample': 0.8835558684167139, 'colsample_bytree': 0.7418481581956126, 'gamma': 0.29214464853521815, 'reg_lambda': 2.9967309097101573}
trial 04  cv_f1=0.5408  trees=10  {'max_depth': 5, 'min_child_weight': 16, 'subsample': 0.7599021346475079, 'colsample_bytree': 0.8542703315240835, 'gamma': 

,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_lambda,cv_f1,mean_best_trees
0,3,19,0.935212,0.906429,0.550243,2.811442,0.582940,46.6
1,3,2,0.936073,0.751305,0.153514,1.573364,0.572046,11.6
2,3,20,0.986578,0.884325,0.915637,1.729554,0.568948,22.2
3,4,3,0.959283,0.761463,0.069544,2.905137,0.565283,5.8
4,3,19,0.969407,0.860419,0.789567,1.983371,0.563624,34.0
5,4,14,0.804265,0.903367,0.640430,8.791110,0.563022,33.8
6,4,19,0.930574,0.938485,0.422571,3.719093,0.560863,28.8
7,4,19,0.897485,0.879892,0.729025,8.105494,0.558859,20.6
8,3,8,0.812274,0.934909,0.742251,7.317376,0.558837,17.4
9,3,1,0.920470,0.825721,0.062845,1.869900,0.557811,8.2


## Stage 2: tree count at a low learning rate

Tree parameters stay fixed. One model is fit on the fit rows at learning rate 0.05. Early stopping watches the eval holdout, and the best round is the tree count. This step does not cross-validate and does not search learning rates.

In [25]:
started = time.perf_counter()
model = make_classifier(
    tree_params,
    learning_rate=STAGE2_LEARNING_RATE,
    n_estimators=STAGE2_N_ESTIMATORS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)
model.fit(X_fit, y_fit, eval_set=[(X_eval, y_eval)], verbose=False)
best_learning_rate = STAGE2_LEARNING_RATE
best_trees = int(model.best_iteration + 1)
print(f"stage 2 finished in {time.perf_counter() - started:.0f}s")
print(f"learning_rate={best_learning_rate}  n_estimators={best_trees}")

stage 2 finished in 1s
learning_rate=0.05  n_estimators=66


## Stage 3: final model

Refit on the fit rows with the stage-1 tree parameters and the stage-2 learning rate and tree count. The eval holdout is not part of this fit.

In [26]:
final_model = make_classifier(
    tree_params,
    learning_rate=best_learning_rate,
    n_estimators=best_trees,
)
final_model.fit(X_fit, y_fit, verbose=False)
print(
    f"fitted {final_model.n_estimators} trees at learning_rate={best_learning_rate} "
    f"on {len(X_fit):,} rows"
)

fitted 66 trees at learning_rate=0.05 on 32,260 rows


## Scores on `train.csv` and `val.csv`

The positive-class threshold is the value on a 0.01 grid that maximizes F1 on the eval holdout. AUC does not use that threshold. `train.csv` includes the fit rows, so its score is partly in-sample. The eval holdout chooses the tree count and the F1 threshold. `val.csv` was not used to fit the model, choose hyperparameters, or choose the threshold.

In [27]:
def predict_proba(features):
    return final_model.predict_proba(features.loc[:, feature_columns])[:, 1]


def f1_threshold(y_true, proba):
    grid = np.linspace(0.01, 0.99, 99)
    scores = [
        f1_score(y_true, proba >= threshold, zero_division=0)
        for threshold in grid
    ]
    best = int(np.argmax(scores))
    return float(grid[best]), float(scores[best])


def binary_scores(y_true, proba, threshold):
    y_true = np.asarray(y_true).astype(int)
    predicted = (proba >= threshold).astype(int)
    return {
        "rows": int(len(y_true)),
        "positives": int(y_true.sum()),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "auc": float(roc_auc_score(y_true, proba)),
    }


eval_proba = predict_proba(X_eval)
threshold, _ = f1_threshold(y_eval.to_numpy(), eval_proba)
print(f"F1 threshold from the eval holdout: {threshold:.2f}")

X_val, y_val = features_and_labels(val)
score_rows = [
    {"split": "train.csv", **binary_scores(y_train, predict_proba(X_train), threshold)},
    {"split": "val.csv", **binary_scores(y_val, predict_proba(X_val), threshold)},
    {"split": "train.csv fit rows", **binary_scores(y_fit, predict_proba(X_fit), threshold)},
    {"split": "train.csv eval holdout", **binary_scores(y_eval, eval_proba, threshold)},
]
score_table = pd.DataFrame(score_rows).set_index("split")
score_table

F1 threshold from the eval holdout: 0.90


,rows,positives,f1,auc
split,,,,
train.csv,40325,810,0.556131,0.961428
val.csv,34315,540,0.611111,0.975936
train.csv fit rows,32260,600,0.591100,0.984379
train.csv eval holdout,8065,210,0.479853,0.896097
